**COMPUTING MONTE CARLO OPTIONS PRICING FOR GOLDMAN SACHS OPTIONS**

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf

In [3]:
gs = yf.Ticker('GS')
expirations = gs.options
print(expirations[:10])

('2026-09-11', '2026-09-18', '2026-09-25', '2026-10-02', '2026-10-09', '2026-10-16', '2026-10-23', '2026-11-20', '2026-12-18', '2027-01-15')


In [4]:
opt_chain = gs.option_chain('2026-11-20')
calls = opt_chain.calls
puts = opt_chain.puts

print(calls.head())
print(calls.columns.tolist())

      contractSymbol             lastTradeDate  strike  lastPrice     bid  \
0  GS261120C00400000 2026-08-03 19:20:31+00:00   400.0     621.28  637.85   
1  GS261120C00430000 2026-08-03 19:21:32+00:00   430.0     591.75  610.60   
2  GS261120C00450000 2026-09-02 15:24:12+00:00   450.0     561.10  588.25   
3  GS261120C00460000 2026-08-03 19:23:14+00:00   460.0     561.63  579.40   
4  GS261120C00470000 2026-08-25 13:30:02+00:00   470.0     579.05  568.65   

      ask  change  percentChange  volume  openInterest  impliedVolatility  \
0  645.75     0.0            0.0     NaN             0           1.083501   
1  614.65     0.0            0.0     NaN             0           1.048711   
2  594.85     0.0            0.0     1.0             4           0.947449   
3  586.35     0.0            0.0     NaN             0           0.986450   
4  576.25     0.0            0.0     1.0             0           0.945069   

   inTheMoney contractSize currency  
0        True      REGULAR      USD 

**CURRENT PRICE**

In [5]:
current_price = gs.history(period='1d')['Close'].iloc[-1]
print(current_price)

1038.6099853515625


**T**

In [17]:
from datetime import datetime

today = datetime.now()
expiry = datetime(2026, 11, 20)
T = (expiry - today).days / 365
print(T)

0.20273972602739726


**IMPLIED VOLATILITY**

In [19]:
from scipy.stats import norm

def black_scholes_call(S, K, T, r, sigma):
    d1 = (np.log(S/K) + (r + sigma**2/2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    call_price = S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)
    return call_price

In [20]:
from scipy.optimize import brentq

def implied_vol_objective(sigma, S, K, T, r, market_price):
    return black_scholes_call(S, K, T, r, sigma) - market_price

implied_vol_calculated = brentq(implied_vol_objective, 0.01, 3.0, args=(current_price, 1040, T, 0.05, 78.00))
print(implied_vol_calculated)

0.39547023238284273


**STIMULATIONG OPTIONS PRICE**

In [22]:
np.random.seed(42)
num_simulations = 10000
Z = np.random.normal(0, 1, num_simulations)

S0 = current_price
sigma = implied_vol_calculated
K = 1040
r = 0.05

simulated_prices = S0 * np.exp((r - sigma**2/2)*T + sigma*np.sqrt(T)*Z)
print(simulated_prices[:10])
print(simulated_prices.shape)

[1128.19013204 1007.57458641 1158.93124297 1354.41220537  990.51662561
  990.51952135 1368.03019245 1183.90828633  949.86870077 1137.43796239]
(10000,)


**PAYOFF COMPUTATION**

In [23]:
payoffs = np.maximum(simulated_prices - K, 0)
print(payoffs[:10])

[ 88.19013204   0.         118.93124297 314.41220537   0.
   0.         328.03019245 143.90828633   0.          97.43796239]
